In [1]:
from pathlib import Path
import re
import unicodedata

import joblib
import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import stopwords
from scipy.sparse import load_npz
from sklearn.metrics.pairwise import cosine_similarity
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk.download("stopwords", quiet=True)

DATA_FOLDER = Path("../../data")
FEATURE_FOLDER = DATA_FOLDER / "features" / "tfidf"

MATRIX_PATH = FEATURE_FOLDER / "tfidf_matrix.npz"
VECTORIZER_PATH = FEATURE_FOLDER / "tfidf_vectorizer.joblib"

DOCUMENTS_PATH = FEATURE_FOLDER / "tfidf_documents.csv"

print(f"Feature folder: {FEATURE_FOLDER.resolve()}")


Feature folder: C:\Works\uts-ir\src\data\features\tfidf


In [2]:
X_main = load_npz(MATRIX_PATH)
vectorizer = joblib.load(VECTORIZER_PATH)
df = pd.read_csv(DOCUMENTS_PATH, low_memory=False)

df.head()

,id,title,abstract,title_translated,abstract_translated,processed_token_text,processed_text
0,2541-3279-1,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,hubungan antara promosi perpustakaan melalui m...,penelitian ini dilatarbelakangi oleh perkemban...,"['hubung', 'promosi', 'pustaka', 'majalah', 'd...",hubung promosi pustaka majalah dinding minat k...
1,2541-3279-2,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,hubungan antara penggunaan search features e-j...,penelitian ini membahas hubungan antara penggu...,"['hubung', 'guna', 'search', 'features', 'jour...",hubung guna search features journals penuh but...
2,2541-3279-3,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,hubungan antara ketersediaan fasilitas perpust...,penelitian ini membahas hubungan antara keters...,"['hubung', 'sedia', 'fasilitas', 'pustaka', 'm...",hubung sedia fasilitas pustaka minat kunjung s...
3,2541-3279-4,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,pengaruh kinerja terhadap layanan prima di cen...,masalah yang menjadi kajian dalam penelitian a...,"['pengaruh', 'kerja', 'layan', 'prima', 'cente...",pengaruh kerja layan prima center of informati...
4,2541-3279-5,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,"['kontribusi', 'literasi', 'informasi', 'mahas...",kontribusi literasi informasi mahasiswa proses...


In [3]:
def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFC", text)
    text = text.casefold()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b", " ", text)
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

stopwords_id = set(stopwords.words("indonesian"))
stemmer = StemmerFactory().create_stemmer()

def preprocess_query(query):
    normalized_query = normalize_text(query)
    tokens = normalized_query.split()
    tokens_no_stopwords = [
        token for token in tokens
        if token not in stopwords_id and len(token) > 1
    ]
    tokens_stemmed = [stemmer.stem(token) for token in tokens_no_stopwords]
    processed_query = " ".join(tokens_stemmed)

    return {
        "normalized_query": normalized_query,
        "tokens": tokens,
        "tokens_no_stopwords": tokens_no_stopwords,
        "tokens_stemmed": tokens_stemmed,
        "processed_query": processed_query,
    }

In [4]:
def search_articles(query, top_k=10):
    if not isinstance(query, str) or not query.strip():
        raise ValueError("Query tidak boleh kosong. Masukkan kata atau frasa pencarian.")
    if not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k harus berupa bilangan bulat positif.")

    query_steps = preprocess_query(query)
    processed_query = query_steps["processed_query"]

    if not processed_query:
        print("Query menjadi kosong setelah preprocessing. Coba gunakan istilah yang lebih spesifik.")
        return pd.DataFrame()

    query_vector = vectorizer.transform([processed_query])
    query_terms = processed_query.split()
    out_of_vocabulary = [term for term in query_terms if term not in vectorizer.vocabulary_]

    if query_vector.nnz == 0:
        print("Tidak ada term query yang ditemukan dalam vocabulary TF-IDF.")
        if out_of_vocabulary:
            print("Term di luar vocabulary:", ", ".join(dict.fromkeys(out_of_vocabulary)))
        return pd.DataFrame()

    scores = cosine_similarity(query_vector, X_main).ravel()
    ranked_indices = np.argsort(scores)[::-1]
    ranked_indices = [idx for idx in ranked_indices if scores[idx] > 0][:top_k]

    if not ranked_indices:
        print("Tidak ditemukan dokumen dengan skor cosine similarity di atas nol.")
        return pd.DataFrame()

    columns_available = set(df.columns)
    display_columns = [
        column for column in ["id", "title", "abstract"]
        if column in columns_available
    ]
    results = df.iloc[ranked_indices][display_columns].copy()
    results.insert(0, "rank", np.arange(1, len(results) + 1))
    results["cosine_similarity"] = scores[ranked_indices]
    results = results.sort_values("cosine_similarity", ascending=False).reset_index(drop=True)

    print(f"Query            : {query}")
    print(f"Processed Query  : {processed_query}")
    print(f"Term ditemukan   : {query_vector.nnz} term(s) dalam vocabulary.")
    if out_of_vocabulary:
        print("Term di luar vocabulary:", ", ".join(dict.fromkeys(out_of_vocabulary)))

    return results

In [5]:
query = "bagaimana cara meningkatkan literasi informasi mahasiswa?"
results = search_articles(query, top_k=10)

# Show all results without truncation
display(results)

Query            : bagaimana cara meningkatkan literasi informasi mahasiswa?
Processed Query  : tingkat literasi informasi mahasiswa
Term ditemukan   : 4 term(s) dalam vocabulary.


,rank,id,title,abstract,cosine_similarity
0,1,2797-9393-24,peran pustakawan referensi perpustakaan uin im...,perpustakaan perguruan tinggi memiliki peran p...,0.629416
1,2,2657-2346-104,ngkat literasi informasi dan literasi akademik...,mahasiswa memiliki peran tidak hanya sebagai p...,0.577379
2,3,2598-7852-71,urgensi kelas literasi informasi bagi mahasisw...,artikel ini bertujuan untuk mengetahui urgensi...,0.558507
3,4,2685-8363-25,strategi peningkatan kompetensi literasi infor...,artikel ini membahas mengenai strategi meningk...,0.553064
4,5,2723-7605-40,kemampuan literasi informasi mahasiswa prodi m...,penelitian ini bertujuan menganalisis kemampua...,0.530804
5,6,2829-9531-18,evaluasi tingkat literasi informasi mahasiswa ...,penelitian ini bertujuan untuk mengidentifikas...,0.526409
6,7,2541-3279-5,kontribusi literasi informasi mahasiswa terhad...,fenomena pesatnya perkembangan ilmu pengetahua...,0.520446
7,8,2715-274x-161,optimalisasi peran perpustakaan dalam meningka...,penelitian ini bertujuan untuk mengeksplorasi ...,0.519991
8,9,2540-9239-182,identifying information literacy skills of new...,perpustakaan sekolah pascasarjana uin syarif h...,0.517218
9,10,2723-7605-39,peran intervensi pustakawan dalam meningkatkan...,peningkatan literasi informasi menjadi sebuah ...,0.514313


In [ ]:
test_queries = [
    "bagaimana cara meningkatkan literasi informasi mahasiswa?",
    "sangkut paut game dalam pendidikan",
    "dampak pandemi covid-19 terhadap pendidikan",
]

# No Truncation of columns in the output display()
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

for test_query in test_queries:
    test_results = search_articles(test_query, top_k=10)
    display(test_results[[
        column for column in ["rank", "id", "title", "abstract", "cosine_similarity"]
        if column in test_results.columns
    ]])

Query            : bagaimana cara meningkatkan literasi informasi mahasiswa?
Processed Query  : tingkat literasi informasi mahasiswa
Term ditemukan   : 4 term(s) dalam vocabulary.


,rank,id,title,abstract,cosine_similarity
0,1,2797-9393-24,peran pustakawan referensi perpustakaan uin imam bonjol padang dalam meningkatkan literasi informasi mahasiswa,"perpustakaan perguruan tinggi memiliki peran penting dalam mendukung proses pengajaran dan penelitian serta memenuhi kebutuhan informasi sivitas akademika terutama mahasiswa. peningkatan literasi informasi mahasiswa sangat diperlukan guna membantu penelusuran dan penggunaan informasi yang efektif dan efisien. kemampuan mahasiswa terkait literasi informasi akan mempermudah dalam membuat karya tulis ilmiah dan pengerjaan tugas kuliah. pustakawan referensi memiliki peran membantu pengguna untuk menemukan informasi yang dibutuhkan secara efektif dan efisien. tujuan penelitian ini untuk dapat mengetahui sejauh mana peran pustakawan referensi dalam peningkatan literasi informasi bagi mahasiswa. metode yang digunakan dalam penulisan artikel ini yaitu metode penelitian deskriptif melalui pendekatan kualitatif. teknik pengumpulan data melalui wawancara, observasi dan studi literatur. penelitian ini menunjukkan pustakawan referensi telah melakukan perannnya dalam meningkatkan literasi informasi, seperti pelatihan dan paket literasi informasi serta berbagai layanan guna meningkatkan kemampuan literasi informasi mahasiswa uin imam bonjol padang.",0.629416
1,2,2657-2346-104,ngkat literasi informasi dan literasi akademik pada mahasiswa uin fatmawati sukarno bengkulu tahun ajaran 2022/2023,"mahasiswa memiliki peran tidak hanya sebagai pembelajar namun juga sebagai penggerak perubahan dan kontrol sosial dalam masyarakat. peran ini menuntut mahasiswa untuk menguasai berbagai kemampuan, termasuk literasi informasi dan literasi ilmiah. kajian terdahulu mengindikasikan adanya kebutuhan mendesak untuk menggali lebih dalam mengenai literasi informasi dan ilmiah di kalangan mahasiswa. penelitian ini bertujuan untuk memberikan gambaran mendalam mengenai tingkat literasi informasi dan ilmiah di kalangan mahasiswa uin fatmawati sukarno bengkulu. desain penelitian ini adalah mix-method dengan menggabungkan data dari kuesioner, focus group discussion, dan tes essay. responden penelitian ini terdiri dari berbagai tingkatan semester dengan total responden berjumlah 80 mahasiswa. hasil penelitian menunjukkan pemahaman literasi informasi dan literasi ilmiah mahasiswa perlu lebih ditingkatkan terutama dalam hal pengaplikasian pengetahuan dalam kegiatan menulis ilmiah.",0.577379
2,3,2598-7852-71,urgensi kelas literasi informasi bagi mahasiswa di perpustakaan universitas muhammadiyah gombong,"artikel ini bertujuan untuk mengetahui urgensi kelas literasi informasi bagi mahasiswa di perpustakaan universitas muhammadiyah gombong. perkembangan teknologi informasi saat ini bukan lagi mengalami evolusi, tetapi sudah menjadi sebuah revolusi informasi yang sangat dahsyat. untuk itu dibutuhkan kemampuan untuk memfilter dari semua informasi yang ada,. perpustakaan sebagai penyedia dan pengelola sekaligus sebagai wahana informasi, maka pustakawan berperan penting dalam memberikan bimbingan penelusuran informasi yang dibutuhkan pemustaka atau mahasiswa. metode penelitian yang digunakan dalam artikel ini yaitu dengan metode deskriptif kualitatif dengan mengkaji secara mendalam terhadap suatu permasalahan. teknik pengumpulan datanya dengan wawancara dan studi literatur. wawancara dilakukan dengan responden mahasiswa, dosen dan pustakawan. dari hasil penelitian dapat disimpulkan bahwa mahasiswa membutuhkan pendampingan dan bimbingan dalam melakukan literasi informasi, sehingga perlu diadakan atau diselenggarakan kelas literasi informasi, hal itu sebagai upaya perpustakaan dalam memberikan layanan literasi informasi. perpustakaan bekerja sama dengan program studi mewajibkan mahasiswa mengikuti kuliah literasi informasi di kelas. dalam kegiatan literasi informasi ini tema yang disampaikan menyesuaikan dengan kebutuhan mahasiswa ,adapun tema- tema literasi yang telah dijalankan antara lain:1} penelusuran ejourn

Query            : sangkut paut game dalam pendidikan
Processed Query  : sangkut paut game didik
Term ditemukan   : 3 term(s) dalam vocabulary.
Term di luar vocabulary: paut


,rank,id,title,abstract,cosine_similarity
0,1,2580-3662-111,game dalam upaya menaikkan citra perpustakaan di kalangan generasi strawbery,"penelitian ini membahas penggunaan game sebagai media untuk meningkatkan citra perpustakaan di kalangan generasi strawberry dan mengubah persepsinya terhadap perpustakaan. dengan memanfaatkan daya tarik game dan mengintegrasikan nilai-nilai literasi, pendidikan, dan interaktivitas, perpustakaan dapat menjadi tempat yang menarik dan edukatif. analisis menunjukkan bahwa game memiliki daya tarik universal bagi generasi strawberry yang terbiasa dengan teknologi dan hiburan digital. dengan desain yang tepat, game dapat menarik minat generasi ini, memungkinkan perpustakaan menjadi lebih sesuai dengan gaya hidupnya. pengintegrasian game ke dalam lingkungan perpustakaan atau gamifikasi memerlukan pendekatan holistik, termasuk dukungan teknis, pendidikan, dan promosi yang tepat. integrasi yang baik akan menciptakan pengalaman kohesif dan berkesinambungan bagi pengunjung. hasil penelitian ini menyimpulkan bahwa penggunaan permainan sebagai alat untuk meningkatkan citra perpustakaan memiliki potensi positif. dengan menggabungkan uniknya perpustakaan sebagai sumber literasi dengan daya tarik game, perpustakaan dapat menjadi tempat menarik, edukatif, dan inovatif bagi generasi muda, membantu nya mengembangkan keterampilan literasi dalam lingkungan yang interaktif dan menyenangkan.",0.460107
1,2,2685-7138-99,hubungan antara artikel mengenai game dan teknologi informasi pada scopus: studi bibliografi,"game sebagai media saat ini sudah diterapkan di beberapa bidang, termasuk pendidikan. implementasi permainan dalam pendidikan dikatakan mampu menawarkan metode pembelajaran yang inovatif dan efektif. di era baru ini, game sangat terkait dengan ilmu informasi. penelitian ini bertujuan untuk mengamati hubungan antara game dan ilmu informasi melalui artikel yang diindeks oleh scopus. penelitian ini adalah kualitatif dengan menggunakan metode blibliografi dalam mengumpulkan data. subjek adalah artikel di situs web scopus yang berisi ""permainan"" dan ""informasi sceince"". data dianalisis menggunakan vosviewer untuk mengungkap koneksi antara mereka. hasil penelitian menunjukkan bahwa antara subjek derek ada lebih banyak hasil breaoder yang dibagi menjadi beberapa sub-subjek. beberapa bagian subjek yang memiliki hubungan dengan informasi yang menjadi jembatan menunjukkan hubungan yang jelas tentang tren subjek sains, termasuk subjek permainan dalam keluarga sains informasi. dapat disimpulkan bahwa ada hubungan kutipan dan tren dalam menulis artikel ilmiah tentang game yang terkait dengan ilmu informasi.",0.403809
2,3,2715-274x-49,pengajaran literasi informasi dengan konsep gamification di perpustakaan perguruan tinggi,"game mungkin bukan hal yang baru bagi perpustakaan di indonesia. bentuk pemanfaatan game di perpustakaan di indonesia biasanya berupa quiz dalam kegiatan orientasi perpustakaan. namun sebenarnya aplikasi game di kegiatan perpustakaan bisa jauh lebih luas daripada itu, termasuk dalam kegiatan pengajaran literasi informasi. prinsip-prinsip game dapat digunakan untuk mengembangkan pengajaran literasi informasi agar kegiatan ini lebih menarik bagi pemustaka dan sekaligus membuat para pemustaka lebih terlibat aktif dalam proses pembelajaran. penerapan prinsip-prinsip game dalam kegiatan yang bukan merupakan kegiatan bermain ini dikenal dengan istilah gamification. gamification merupakan tren yang kian meluas dan kian mendapat tempat, tidak hanya di dunia pendidikan, pengembangan personal maupun dunia bisnis. mengingat manfaat yang bisa dipetik dari gamification, maka perpustakaan juga perlu belajar bagaimana mengaplikasikannya dalam dunia perpustakaan, terutama untuk pengembangan fungsi pendidikan, yakni pengajaran literasi informasi",0.373403
3,4,2502-6003-68,peningkatan pengetahuan “dampak negatif game online” melalui media poster pada siswa smpn 79 jakarta,"penelitian ini bertujuan untuk menganalisis 

Query            : dampak pandemi covid-19 terhadap pendidikan
Processed Query  : dampak pandemi covid didik
Term ditemukan   : 4 term(s) dalam vocabulary.


,rank,id,title,abstract,cosine_similarity
0,1,2775-8761-38,dampak rendahnya perpustakaan di masa pandemi covid-19 pada siswa smpn 2 raha,"masalah yang dihadapi sekolah saat ini adalah masa pandemic covid 19 yang menghambat proses strategi perpustakaan sekolah hampir tidak ada yang menyangka, wajah pendidikan akan berubah drastis akibat pandemi covid 19. penelitian ini bertujuan untuk mengetahui dampak rendahnya pemanfaatan perpustakaan pimasa pandemi covid-19 pada siswa smpn 2 raha. metode penelitian yang digunakan adalah deskripsi kualitatif, informan dalam penelitian sebanyak lima orang. teknik pengumpulan data dilakukan dengan melalui observasi, wawancara dan dokumentasi. hasil penelitian ini menunjukkan bahwa dampak rendah pemanfaatan perpustakaan di masa pandemi covid-19 pada smpn 2 raha ada beberapa dampak sesuai dengan menurut sutarno (2003). dampak rendahnya pemanfaatan perpustakaan di masa pandemi covid-19 ada 4 dampak yaitu pemeliharaan bahan pustaka, kebersihan perpustakaan, kerapian koleksi, penataan koleksi. pengelola perpustakaan dan pemustaka merasa tehalang dengan adanya covid-19 sehingga mereka tidak bisa menata perpustakaan dengan baik dan pengunjung juga tidak bisa berkunjung ke perpustakaaan seperti biasanya. penelitian ini diharapkan dapat menjadi rujukan sebagai bahan informasi bagi peneliti selanjutnya yang akan meneliti topik yang berkaitan dengan dampakrendanya pemanfaatan perpustakaan di masa pandemi covid-19.",0.590535
1,2,2685-7138-94,wacana pandemi covid-19 terhadap penutupan perpustakaan,"coronavirus disease (covid-19) telah ditetapkan statusnya sebagai pandemi oleh who karena penyebarannya ke berbagai negara dalam waktu yang sama. perpustakaan pun tidak luput dari dampak wacana pandemi covid-19 karena harus ditutup sebagai upaya yang dilakukan secara bersama-sama dalam memutus rantai penyebaran covid-19. melalui studi literatur terutama dari jurnal ilmiah, penulis mencoba mengumpulkan berbagai informasi terkait dengan penutupan perpustakaan dikarenakan krisis, untuk kemudian dapat diketahui hal-hal yang dapat dilakukan perpustakaan di tengah pandemi covid-19. hasil studi menunjukkan selama masa pandemi covid- 19 perpustakaan harus menentukan kebijakan terkait layanan seperti buka atau tidak dan pembebasan denda. perlunya mempertimbangkan mengenai tindakan disinfektan terhadap gedung dan koleksi secara keseluruhan. perpustakaan harus menjadi pusat informasi tepercaya sebagai upaya memperlambat penyebaran virus dan informasi yang salah. perpustakaan juga seyogyanya menyediakan sumber informasi yang dapat diakses pemustaka melalui electronic resources. perpustakaan dan pustakawan selama masa pandemi dapat terlibat sebagai gugus tugas covid-19 yakni gedung perpustakaan dapat digunakan sebagai gudang logistik untuk pendistribusian masker, alat uji dan alat pelindung diri (apd). terakhir, perpustakaan dapat mengadakan program yang bermanfaat untuk pemustaka melalui konsep layanan “boutique” dan webinar. hasil studi dapat dijadikan panduan dan lesson learned ke depannya untuk dapat diterapkan oleh perpustakaan dan pustakawan saat masa krisis.",0.451114
2,3,2685-8363-130,peran teknologi informasi dalam layanan online perpustakaan masa pandemi covid-19,"pandemi covid-19 telah menyerang berbagai lapisan masyarakat dari seluruh belahan dunia, termasuk indonesia. banyak di antara masyarakat yang terkena dampak covid-19 mulai beradaptasi terhadap keadaan baru tersebut dengan berbagai cara. perpustakaan sebagai sumber informasi dituntut untuk menyesuaikan kebutuhan pada pemustakanya. penulis menggunakan studi literatur atau kajian pustaka dalam menyusun tulisan ini. peran teknologi di masa pandemi memberikan kemudahan dalam memanfaatkan perpustakaan. pustakawan melakukan adaptasi layanan melalui online seperti: layanan konsultasi, layanan penelusuran sumber digital, layanan pengecekan plagiarism, dan membangun digital library. peran teknologi di perpustakaan dalam memberikan layanan selama pandemi covid-19 mem